# 03 — Análisis de los detectores

Analiza los 10 modelos de `02_entrenamiento_detectores` (TranAD y TranAD+ con 5 semillas cada uno). **No entrena nada y no necesita GPU:** trabaja con los errores por variable que guardó el notebook 02.

Calcula todas las cifras del Experimento 1 (apartado 5.1 de la memoria) y prepara las alertas que usan los Experimentos 2 y 3. Las figuras de la memoria se generan en `04_experimento_1` a partir de lo que se guarda aquí.

| Sección | Qué calcula | Apartado de la memoria |
|---|---|---|
| 1–3 | Entorno, parámetros y puntuaciones de los 10 modelos | — |
| 4 | Concordancia de las alertas (índice de Jaccard) y criterio de decisión de H1 | 5.1.2 |
| 5 | Alertas de consenso de cada arquitectura y alertas comunes | 5.1.3 |
| 6 | Señal protagonista y familia de cada alerta de consenso | 5.1.3 |
| 7 | Umbral POT: tasa de alarma y detección de los valores imposibles | 5.1.4 |
| 8 | Pérdida de entrenamiento por época | 5.1.1 |
| 9 | Alertas comunes: contribución de cada variable, metadatos y resumen por familia | 4.4.3 y 4.4.4 |
| 10 | Guardado y resumen de las comprobaciones con la memoria | — |

**Comprobaciones con la memoria:** cada cifra que aparece en la memoria se compara con la calculada y se marca como `coincide` o `distinto`. Con otros datos no coinciden, y es normal.

**Salidas** (en `RUTA_ANALISIS_DETECTORES`):
- `jaccard_parejas.csv`, `criterio_decision.csv`, `consenso_k100.npz`, `familias_consenso_k100.csv`, `pot_por_corrida.csv` y `perdidas_por_epoca.csv`: cifras del Experimento 1.
- `alertas_comunes_k100.csv` y `contribuciones_alertas_comunes.npz`: alertas comunes a las dos arquitecturas, entrada de los Experimentos 2 y 3.
- `percentiles_99_no_cero_train.npy`: valor alto habitual de cada variable en entrenamiento.
- `comprobaciones_memoria.csv`: resultado de todas las comprobaciones.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import gc
from itertools import combinations, product

import joblib
import numpy as np
import pandas as pd
import torch  # solo para leer la pérdida guardada en los checkpoints, en CPU

from config import (RUTA_PREPARADOS, RUTA_DETECTORES, RUTA_ANALISIS_DETECTORES,
                    NOMBRES_FEATURES_COMPLETO, IDX_CONTADORES, IDX_GAP, FAMILIA_DE_VARIABLE, ORDEN_FAMILIAS)
from datos import desescalar
from evaluacion import umbral_pot, calcular_porcentaje_sobre_umbral, score_desde_errores
from senal_protagonista import calcular_percentiles_referencia, identificar_senal_protagonista

os.makedirs(RUTA_ANALISIS_DETECTORES, exist_ok=True)
print("Errores de los detectores:", RUTA_DETECTORES)
print("Resultados en:", RUTA_ANALISIS_DETECTORES)

## 2. Parámetros

Fijados antes de ver ningún resultado (apartado 2.2.2 de la memoria):

- **k = 100** alertas principales por modelo, cerca del número de alertas que da el umbral POT (alrededor del 0,1 % de `test`). Se repite con 50, 200 y 500 para comprobar que la conclusión no depende de k.
- **Consenso:** una ventana es alerta estable de una arquitectura si está entre las k principales en al menos **3 de sus 5** semillas.
- **POT:** q = 10⁻³ y percentil inicial 98, calibrado en `val` (apartado 4.3.2).
- **Criterio de decisión de H1** (descriptivo, porque las parejas comparten semillas y no son independientes). La referencia es la pareja de una misma arquitectura que menos coincide:
  - *evidencia fuerte:* todas las parejas TranAD–TranAD+ coinciden menos que esa referencia;
  - *evidencia débil:* al menos la mitad coinciden menos (la mediana está por debajo).
- **Familias de variables:** las de `config.FAMILIA_DE_VARIABLE` (FAM_1 a FAM_4). Un error y sus recuperaciones están en la misma familia.

In [ ]:
# --- Parámetros del análisis, fijados antes de ver resultados ---
SEMILLAS = [0, 7, 42, 123, 777]
MODELOS = {"tranad": "TranAD", "tranadplus": "TranAD+"}
CORRIDAS = [f"{m}_s{s}" for m in MODELOS for s in SEMILLAS]  # tranad_s0 ... tranad_s777, tranadplus_s0 ...

K_PRINCIPAL = 100
K_VALORES = [50, 100, 200, 500]
MINIMO_CONSENSO = 3
Q_POT = 1e-3
NIVEL_INICIAL_POT = 0.98
PERCENTIL_REFERENCIA = 99
N_EPOCAS = 15
CONJUNTOS = ["val", "test", "valores_imposibles"]
SENALES = list(NOMBRES_FEATURES_COMPLETO)


SIN_PROTAGONISTA = "Sin protagonista"
ORDEN_FAMILIAS_TABLAS = ORDEN_FAMILIAS + [SIN_PROTAGONISTA]  # orden de filas y columnas en las tablas

# Registro de las comparaciones con las cifras de la memoria
COMPROBACIONES = []


def comprobar(descripcion, calculado, memoria, tolerancia=0.0):
    # Compara una cifra calculada con la de la memoria y la anota en el registro.
    coincide = abs(float(calculado) - float(memoria)) <= tolerancia
    COMPROBACIONES.append({"comprobacion": descripcion, "calculado": calculado, "memoria": memoria,
                           "resultado": "coincide" if coincide else "distinto"})
    print(f"  {'coincide' if coincide else 'DISTINTO'}: {descripcion} = {calculado} (memoria: {memoria})")

## 3. Puntuaciones de los 10 modelos

La puntuación de anomalía de cada ventana se calcula a partir de los errores por variable que guardó el notebook 02, con los pesos α y β guardados en el propio fichero (ecuación 1 de la memoria). En el notebook 02 se comprobó que coincide con la que da el modelo directamente.

In [ ]:
# --- Carga de los errores por variable y cálculo de las puntuaciones ---
puntuaciones = {}  # corrida -> {conjunto: puntuación de cada ventana}
errores_test = {}  # corrida -> (alfa, beta, e1, e2) de test, para las contribuciones de la sección 9
for corrida in CORRIDAS:
    with np.load(f"{RUTA_DETECTORES}/errores/{corrida}.npz") as fichero:
        alfa, beta = float(fichero["alfa"]), float(fichero["beta"])
        puntuaciones[corrida] = {c: score_desde_errores(fichero[f"{c}_e1"], fichero[f"{c}_e2"], alfa, beta)
                                 for c in CONJUNTOS}
        errores_test[corrida] = (alfa, beta, fichero["test_e1"], fichero["test_e2"])
    print(f"{corrida:16s} α = {alfa}, β = {beta} | test: {len(puntuaciones[corrida]['test']):,} ventanas")

N_TEST = len(puntuaciones[CORRIDAS[0]]["test"])
print(f"\n{len(puntuaciones)} modelos cargados")
comprobar("ventanas de prueba", N_TEST, 88_126)

## 4. Concordancia de las alertas (índice de Jaccard)

Para cada modelo se toman sus k ventanas de `test` con mayor puntuación. Para cada pareja de modelos, el índice de Jaccard es el número de alertas compartidas dividido por el número de alertas distintas: 1 si las listas son iguales y 0 si no comparten ninguna.

| Grupo | Qué compara | Parejas |
|---|---|---|
| `dentro_tranad` | Dos semillas de TranAD | 10 |
| `dentro_tranadplus` | Dos semillas de TranAD+ | 10 |
| `entre_modelos` | Una semilla de TranAD con una de TranAD+ | 25 |

Los dos primeros grupos miden cuánto cambian las alertas solo por el azar del entrenamiento. Son la referencia con la que se compara el tercero.

In [ ]:
# --- Índice de Jaccard de cada pareja de modelos, para cada k ---
def top_k(scores, k):
    # Índices de las k ventanas con mayor puntuación (orden estable si hay empates)
    return set(np.argsort(-scores, kind="stable")[:k])


def jaccard(a, b):
    # Alertas compartidas divididas por alertas distintas
    return len(a & b) / len(a | b)


filas = []
for k in K_VALORES:
    alertas = {c: top_k(puntuaciones[c]["test"], k) for c in CORRIDAS}
    for modelo in MODELOS:  # parejas de una misma arquitectura: efecto del azar del entrenamiento
        for s1, s2 in combinations(SEMILLAS, 2):
            c1, c2 = f"{modelo}_s{s1}", f"{modelo}_s{s2}"
            filas.append({"k": k, "grupo": f"dentro_{modelo}", "corrida_1": c1, "corrida_2": c2,
                          "jaccard": jaccard(alertas[c1], alertas[c2])})
    for s1, s2 in product(SEMILLAS, SEMILLAS):  # parejas entre arquitecturas: arquitectura + azar
        c1, c2 = f"tranad_s{s1}", f"tranadplus_s{s2}"
        filas.append({"k": k, "grupo": "entre_modelos", "corrida_1": c1, "corrida_2": c2,
                      "jaccard": jaccard(alertas[c1], alertas[c2])})

df_jaccard = pd.DataFrame(filas)
resumen_jaccard = (df_jaccard.groupby(["k", "grupo"])["jaccard"]
                   .agg(["count", "min", "median", "mean", "max"]).round(3))
display(resumen_jaccard)

medianas_k100 = df_jaccard[df_jaccard["k"] == K_PRINCIPAL].groupby("grupo")["jaccard"].median()
comprobar("Jaccard mediano dentro de TranAD (k = 100)", round(float(medianas_k100["dentro_tranad"]), 3), 0.544)
comprobar("Jaccard mediano dentro de TranAD+ (k = 100)", round(float(medianas_k100["dentro_tranadplus"]), 3), 0.581)
comprobar("Jaccard mediano entre arquitecturas (k = 100)", round(float(medianas_k100["entre_modelos"]), 3), 0.575)

### 4.1 Criterio de decisión de H1

In [ ]:
# --- Aplicación del criterio de decisión para cada k ---
filas = []
for k in K_VALORES:
    sub = df_jaccard[df_jaccard["k"] == k]
    dentro = sub[sub["grupo"].str.startswith("dentro")]["jaccard"]
    entre = sub[sub["grupo"] == "entre_modelos"]["jaccard"]
    filas.append({
        "k": k,
        "min_dentro": round(dentro.min(), 3),        # pareja de una misma arquitectura que menos coincide
        "mediana_entre": round(entre.median(), 3),
        "max_entre": round(entre.max(), 3),
        "criterio_fuerte": entre.max() < dentro.min(),     # todas las parejas entre arquitecturas por debajo
        "criterio_debil": entre.median() < dentro.min(),   # al menos la mitad por debajo
    })
df_criterio = pd.DataFrame(filas)
display(df_criterio)

# En la memoria no se cumple ningún criterio con ningún k: H1 queda refutada
comprobar("valores de k con algún criterio cumplido",
          int((df_criterio["criterio_fuerte"] | df_criterio["criterio_debil"]).sum()), 0)

## 5. Alertas de consenso

Para cada arquitectura, una ventana es **alerta de consenso** si está entre las 100 principales en al menos 3 de sus 5 semillas. Así se quita el efecto del azar del entrenamiento.

Después se comparan las dos arquitecturas:
- `comun`: alertas de consenso de las dos. Son las que se usan en los Experimentos 2 y 3.
- `solo_tranad` y `solo_tranadplus`: alertas de consenso de una sola arquitectura.

In [ ]:
# --- Alertas de consenso de cada arquitectura y comparación entre ellas ---
def alertas_consenso(modelo, k, minimo):
    # Ventanas que están en el top-k de al menos 'minimo' semillas de la arquitectura
    conteo = np.zeros(N_TEST, dtype=int)
    for semilla in SEMILLAS:
        conteo[list(top_k(puntuaciones[f"{modelo}_s{semilla}"]["test"], k))] += 1
    return set(np.where(conteo >= minimo)[0])


consenso = {m: alertas_consenso(m, K_PRINCIPAL, MINIMO_CONSENSO) for m in MODELOS}
comun = consenso["tranad"] & consenso["tranadplus"]
solo_tranad = consenso["tranad"] - consenso["tranadplus"]
solo_tranadplus = consenso["tranadplus"] - consenso["tranad"]

comprobar("alertas de consenso de TranAD", len(consenso["tranad"]), 96)
comprobar("alertas de consenso de TranAD+", len(consenso["tranadplus"]), 88)
comprobar("alertas comunes", len(comun), 74)
comprobar("alertas solo de TranAD", len(solo_tranad), 22)
comprobar("alertas solo de TranAD+", len(solo_tranadplus), 14)

## 6. Señal protagonista y familia

La **señal protagonista** de una alerta es la variable que más supera su valor alto habitual en entrenamiento: el percentil 99 de sus valores distintos de cero (con tantos ceros, el percentil sobre todos los valores sería casi siempre 0). Si ninguna lo supera, la alerta queda **sin protagonista**. La regla depende solo de los datos y no de ningún modelo, así que no favorece a ninguna arquitectura.

### 6.1 Valor alto habitual de cada variable en `train`

Es el paso que más memoria consume: las ventanas de entrenamiento se pasan a valores reales y se liberan en cuanto se calcula el percentil.

In [ ]:
# --- Percentil 99 de los valores distintos de cero de cada variable en train ---
escalador_contadores = joblib.load(f"{RUTA_PREPARADOS}/escalador_contadores.pkl")
escalador_gap = joblib.load(f"{RUTA_PREPARADOS}/escalador_gap.pkl")


def a_valores_reales(X_escalado):
    # Deshace el escalado del notebook 01 (MinMax y logaritmo)
    return desescalar(X_escalado, escalador_contadores, escalador_gap, IDX_CONTADORES, IDX_GAP)


with np.load(f"{RUTA_PREPARADOS}/ventanas_escaladas.npz") as datos:
    X_train_real = a_valores_reales(datos["train"])
    X_test_esc = datos["test"]
percentiles_99 = calcular_percentiles_referencia(X_train_real, percentil=PERCENTIL_REFERENCIA)
del X_train_real
gc.collect()

for nombre, valor in zip(SENALES, percentiles_99):
    print(f"  {nombre:9s} {valor:,.1f}")

### 6.2 Familia de cada alerta de consenso

In [ ]:
# --- Señal protagonista y familia de las alertas de consenso ---
pertenencia = ([(i, "comun") for i in comun] + [(i, "solo_tranad") for i in solo_tranad]
               + [(i, "solo_tranadplus") for i in solo_tranadplus])

filas = []
for indice, grupo in sorted(pertenencia):
    ventana_real = a_valores_reales(X_test_esc[indice:indice + 1])[0]  # (12, 13) en valores reales
    r = identificar_senal_protagonista(ventana_real, percentiles_99, SENALES)
    filas.append({
        "indice_en_x_test": indice,
        "grupo": grupo,
        "columna_protagonista": r["columna_protagonista"],
        "protagonista_valido": r["protagonista_valido"],
        "familia": FAMILIA_DE_VARIABLE[r["columna_protagonista"]] if r["protagonista_valido"] else SIN_PROTAGONISTA,
    })
df_familias = pd.DataFrame(filas)

conteo_familias = (pd.crosstab(df_familias["grupo"], df_familias["familia"])
                   .reindex(index=["comun", "solo_tranad", "solo_tranadplus"], columns=ORDEN_FAMILIAS_TABLAS, fill_value=0))
display(conteo_familias)

# En la memoria solo se interpretan las comunes: las exclusivas no, porque H1 no se cumple
comprobar("alertas comunes de la familia FAM_2", int(conteo_familias.loc["comun", "FAM_2"]), 36)
comprobar("alertas comunes de la familia FAM_4", int(conteo_familias.loc["comun", "FAM_4"]), 24)

## 7. Umbral POT: tasa de alarma y detección de los valores imposibles

Para cada modelo se calcula el umbral POT con las puntuaciones de `val` y se aplica a `test` y a las 885 ventanas con valores imposibles. La tasa de alarma que interesa es la de `test`: la de `val` queda fijada al calibrar ahí el umbral.

In [ ]:
# --- Umbral POT de cada modelo y resultados con ese umbral ---
filas = []
for corrida in CORRIDAS:
    p = puntuaciones[corrida]
    umbral = umbral_pot(p["val"], q=Q_POT, nivel_inicial=NIVEL_INICIAL_POT)
    modelo, semilla = corrida.rsplit("_s", 1)
    filas.append({
        "corrida": corrida, "modelo": modelo, "semilla": int(semilla),
        "umbral_pot": umbral,
        "n_alertas_test": int((p["test"] > umbral).sum()),
        "alarma_val_pct": calcular_porcentaje_sobre_umbral(p["val"], umbral),
        "alarma_test_pct": calcular_porcentaje_sobre_umbral(p["test"], umbral),
        "deteccion_valores_imposibles_pct": calcular_porcentaje_sobre_umbral(p["valores_imposibles"], umbral),
    })
df_pot = pd.DataFrame(filas)
display(df_pot.round(4))

print("Media y desviación típica de las 5 semillas:")
resumen_pot = (df_pot.groupby("modelo")[["n_alertas_test", "alarma_test_pct", "deteccion_valores_imposibles_pct"]]
               .agg(["mean", "std"]).round(3))
display(resumen_pot)

media = df_pot.groupby("modelo")[["alarma_test_pct", "deteccion_valores_imposibles_pct"]].mean()
sin_s42 = df_pot[(df_pot["modelo"] == "tranadplus") & (df_pot["semilla"] != 42)]["deteccion_valores_imposibles_pct"].mean()
s42 = df_pot.loc[df_pot["corrida"] == "tranadplus_s42", "deteccion_valores_imposibles_pct"].iloc[0]
comprobar("tasa de alarma en test, TranAD (media %)", round(float(media.loc["tranad", "alarma_test_pct"]), 3), 0.137)
comprobar("tasa de alarma en test, TranAD+ (media %)", round(float(media.loc["tranadplus", "alarma_test_pct"]), 3), 0.121)
comprobar("detección de valores imposibles, TranAD (media %)",
          round(float(media.loc["tranad", "deteccion_valores_imposibles_pct"]), 1), 81.3)
comprobar("detección de valores imposibles, TranAD+ (media %)",
          round(float(media.loc["tranadplus", "deteccion_valores_imposibles_pct"]), 1), 73.4)
comprobar("detección de valores imposibles, TranAD+ semilla 42 (%)", round(float(s42), 1), 40.5)
comprobar("detección de valores imposibles, TranAD+ sin la semilla 42 (media %)", round(float(sin_s42), 1), 81.7)

## 8. Pérdida de entrenamiento por época

Se lee la pérdida guardada en cada checkpoint del notebook 02 (10 modelos × 15 épocas). El peso de cada decodificador en la pérdida cambia con la época (ecuación 4 de la memoria), así que la curva no se lee como una bajada continua: lo que tiene sentido es **comparar semillas en la misma época**. Para eso se divide la pérdida de cada semilla por la mediana de su arquitectura en esa época: 1 es lo típico.

In [ ]:
# --- Pérdida de cada época, leída de los checkpoints (en CPU) ---
filas = []
for corrida in CORRIDAS:
    modelo, semilla = corrida.rsplit("_s", 1)
    for epoca in range(1, N_EPOCAS + 1):
        checkpoint = torch.load(f"{RUTA_DETECTORES}/checkpoints/{corrida}/checkpoint_epoca_{epoca}.pt",
                                map_location="cpu", weights_only=False)
        filas.append({"modelo": modelo, "semilla": int(semilla), "epoca": epoca,
                      "perdida": float(checkpoint["perdida"])})
        del checkpoint  # solo interesa la pérdida
df_perdidas = pd.DataFrame(filas)
print(f"Pérdidas leídas: {len(df_perdidas)} (esperadas: {len(CORRIDAS) * N_EPOCAS})")

# Pérdida de cada semilla dividida por la mediana de su arquitectura en la misma época
df_perdidas["relativa_mediana_epoca"] = (df_perdidas["perdida"]
                                         / df_perdidas.groupby(["modelo", "epoca"])["perdida"].transform("median"))
for modelo, nombre in MODELOS.items():
    print(f"\n{nombre}: pérdida / mediana de la época")
    display(df_perdidas[df_perdidas["modelo"] == modelo]
            .pivot(index="epoca", columns="semilla", values="relativa_mediana_epoca").round(2))

## 9. Alertas comunes

Las alertas comunes a las dos arquitecturas son la entrada de los Experimentos 2 y 3. Para cada una se calcula:

- **Contribución de cada variable al error de reconstrucción.** Cada modelo reparte su error entre las 13 variables, en porcentaje de su propio error total, y se toma la **mediana de los 10 modelos**. Al ser porcentajes, son comparables entre arquitecturas aunque cada una use pesos α y β distintos.
- **Puntuación de cada arquitectura:** mediana de sus 5 semillas. No se combinan las dos arquitecturas, porque calculan la puntuación con pesos distintos.
- **Metadatos** de la ventana (dispositivo seudonimizado, sesión y momento de inicio), para poder localizarla.

### 9.1 Contribución de cada variable

In [ ]:
# --- Contribución de cada variable al error, por modelo, por arquitectura y de consenso ---
indices = np.array(sorted(comun), dtype=int)
N_COMUNES = len(indices)

reparto = {}  # corrida -> (N_COMUNES, 13), en % del error total del modelo
score = {}    # corrida -> (N_COMUNES,)
for corrida in CORRIDAS:
    alfa, beta, e1, e2 = errores_test[corrida]
    e = (alfa * e1[indices] + beta * e2[indices]).astype(np.float64)  # error por variable (ecuación 1)
    total = e.sum(axis=1, keepdims=True)
    reparto[corrida] = np.divide(100 * e, total, out=np.zeros_like(e), where=total > 0)
    score[corrida] = e.mean(axis=1)

todas = np.stack([reparto[c] for c in CORRIDAS])  # (10, N_COMUNES, 13)
contrib_consenso = np.median(todas, axis=0)
contrib_arq = {m: np.median(np.stack([reparto[f"{m}_s{s}"] for s in SEMILLAS]), axis=0) for m in MODELOS}
score_arq = {m: np.median(np.stack([score[f"{m}_s{s}"] for s in SEMILLAS]), axis=0) for m in MODELOS}


def senales_principales(contrib, n=1):
    # Nombres de las n variables con mayor contribución en cada ventana
    orden = np.argsort(-contrib, axis=1, kind="stable")[:, :n]
    return [[SENALES[j] for j in fila] for fila in orden]


top1_consenso = [t[0] for t in senales_principales(contrib_consenso)]
top1_arq = {m: [t[0] for t in senales_principales(contrib_arq[m])] for m in MODELOS}

# ¿Atribuyen las dos arquitecturas la alerta a la misma variable?
coincide_top1 = np.array([a == b for a, b in zip(top1_arq["tranad"], top1_arq["tranadplus"])])
top3 = {m: senales_principales(contrib_arq[m], 3) for m in MODELOS}
solape_top3 = np.mean([len(set(a) & set(b)) / 3 for a, b in zip(top3["tranad"], top3["tranadplus"])])
print(f"Misma variable de mayor contribución en las dos arquitecturas: "
      f"{coincide_top1.sum()} de {N_COMUNES} ({coincide_top1.mean():.1%})")
print(f"Solape medio de las 3 variables de mayor contribución: {solape_top3:.1%}")

### 9.2 Señal protagonista y metadatos

Los metadatos se leen de `metadatos_test.parquet`, que el notebook 01 guardó en el mismo orden que las ventanas de `test`.

In [ ]:
# --- Protagonista y familia de cada alerta común, y sus metadatos ---
X_real_comunes = a_valores_reales(X_test_esc[indices])  # (N_COMUNES, 12, 13)
protagonista, valido, familia = [], [], []
for ventana in X_real_comunes:
    r = identificar_senal_protagonista(ventana, percentiles_99, SENALES)
    protagonista.append(r["columna_protagonista"] if r["protagonista_valido"] else "—")
    valido.append(bool(r["protagonista_valido"]))
    familia.append(FAMILIA_DE_VARIABLE[r["columna_protagonista"]] if r["protagonista_valido"] else SIN_PROTAGONISTA)

# ¿Coincide la variable de mayor contribución del detector con la protagonista de los datos?
coincide_detector_datos = np.array([d == p for d, p in zip(top1_consenso, protagonista)])
print(f"Variable de mayor contribución = señal protagonista: {coincide_detector_datos.sum()} de {N_COMUNES} "
      f"({sum(valido)} con protagonista)")

meta_test = pd.read_parquet(f"{RUTA_PREPARADOS}/metadatos_test.parquet")
assert len(meta_test) == N_TEST, "Los metadatos no tienen el mismo número de ventanas que test"
metadatos = meta_test.loc[indices, ["device_pseudo_id", "session_id", "window_id", "inicio_ventana"]].reset_index(drop=True)

# Solo recuentos: los identificadores de dispositivo no se muestran
ventanas_por_dispositivo = metadatos["device_pseudo_id"].value_counts()
print(f"Dispositivos distintos: {len(ventanas_por_dispositivo)} | "
      f"con más de una alerta: {(ventanas_por_dispositivo > 1).sum()} | "
      f"máximo de alertas en un dispositivo: {ventanas_por_dispositivo.max()}")

### 9.3 Resumen por familia

Para cada familia: número de alertas, de dispositivos distintos y la mediana del valor máximo de algunas variables en la ventana. En la familia FAM_4, V01 indica cuánto tiempo llevaba el dispositivo sin reportar antes de la ventana.

In [ ]:
# --- Alertas, dispositivos y valores típicos por familia ---
SENALES_RESUMEN = ["V01", "S01ERR", "S03", "S03L01", "S04"]
maximos = np.abs(X_real_comunes).max(axis=1)  # (N_COMUNES, 13): máximo de cada variable en cada ventana
df_resumen = pd.DataFrame({"familia": familia, "device_pseudo_id": metadatos["device_pseudo_id"]})
for s in SENALES_RESUMEN:
    df_resumen[s] = maximos[:, SENALES.index(s)]

resumen_familias = (df_resumen.groupby("familia")
                    .agg(alertas=("familia", "size"), dispositivos=("device_pseudo_id", "nunique"),
                         **{f"mediana_max_{s}": (s, "median") for s in SENALES_RESUMEN})
                    .reindex([f for f in ORDEN_FAMILIAS_TABLAS if f in set(familia)]))
resumen_familias["% alertas"] = (100 * resumen_familias["alertas"] / N_COMUNES).round(1)
display(resumen_familias.round(1))

# La memoria habla de «unos 11 días» sin reportar en la familia FAM_4
if "FAM_4" in resumen_familias.index:
    dias = resumen_familias.loc["FAM_4", "mediana_max_V01"] / (60 * 24)
    comprobar("mediana de días sin reportar en la familia FAM_4", round(float(dias)), 11)

## 10. Guardado

- `alertas_comunes_k100.csv`: una fila por alerta común, con sus metadatos, familia, señal protagonista, puntuación de cada arquitectura, variables de mayor contribución y contribución de consenso de cada variable (`contrib_<variable>`). La columna `protagonista_v01_v02` marca las alertas cuya protagonista es V01 o V02, que el Experimento 2 excluye porque el modelo de lenguaje no recibe esas variables (apartado 4.4.3).
- `contribuciones_alertas_comunes.npz`: contribuciones completas (de consenso, por arquitectura y por modelo).
- El resto de ficheros, con las cifras del Experimento 1 para `04_experimento_1`.

In [ ]:
# --- Guardado de los resultados ---
R = RUTA_ANALISIS_DETECTORES
df_jaccard.to_csv(f"{R}/jaccard_parejas.csv", index=False)
resumen_jaccard.to_csv(f"{R}/jaccard_resumen.csv")
df_criterio.to_csv(f"{R}/criterio_decision.csv", index=False)
df_familias.to_csv(f"{R}/familias_consenso_k{K_PRINCIPAL}.csv", index=False)
df_pot.to_csv(f"{R}/pot_por_corrida.csv", index=False)
df_perdidas.to_csv(f"{R}/perdidas_por_epoca.csv", index=False)
np.save(f"{R}/percentiles_99_no_cero_train.npy", percentiles_99)

# Índices (posición en test) de las alertas de consenso, para no recalcularlos en otros notebooks
np.savez(f"{R}/consenso_k{K_PRINCIPAL}.npz",
         consenso_tranad=np.array(sorted(consenso["tranad"])),
         consenso_tranadplus=np.array(sorted(consenso["tranadplus"])),
         comun=np.array(sorted(comun)),
         solo_tranad=np.array(sorted(solo_tranad)),
         solo_tranadplus=np.array(sorted(solo_tranadplus)))

tabla_comunes = pd.concat([
    pd.DataFrame({"indice_en_x_test": indices}),
    metadatos,
    pd.DataFrame({
        "familia": familia,
        "senal_protagonista": protagonista,
        "protagonista_valido": valido,
        "score_mediano_tranad": score_arq["tranad"],
        "score_mediano_tranadplus": score_arq["tranadplus"],
        "top1_consenso": top1_consenso,
        "top1_tranad": top1_arq["tranad"],
        "top1_tranadplus": top1_arq["tranadplus"],
        "coincide_top1_arquitecturas": coincide_top1,
        "protagonista_v01_v02": [p in ("V01", "V02") for p in protagonista],
    }),
    pd.DataFrame(contrib_consenso.round(3), columns=[f"contrib_{s}" for s in SENALES]),
], axis=1)
tabla_comunes.to_csv(f"{R}/alertas_comunes_k{K_PRINCIPAL}.csv", index=False)

np.savez(f"{R}/contribuciones_alertas_comunes.npz",
         indices=indices, senales=np.array(SENALES), contrib_consenso=contrib_consenso,
         contrib_tranad=contrib_arq["tranad"], contrib_tranadplus=contrib_arq["tranadplus"],
         reparto_por_modelo=todas, modelos=np.array(CORRIDAS))

pd.DataFrame(COMPROBACIONES).to_csv(f"{R}/comprobaciones_memoria.csv", index=False)

print("Guardado en", R)
for nombre in sorted(os.listdir(R)):
    print("  ", nombre)

### 10.1 Resumen de las comprobaciones con la memoria

In [ ]:
# --- Resumen de todas las comprobaciones ---
df_comprobaciones = pd.DataFrame(COMPROBACIONES)
distintas = df_comprobaciones[df_comprobaciones["resultado"] != "coincide"]
print(f"Comprobaciones: {len(df_comprobaciones)} | coinciden: {len(df_comprobaciones) - len(distintas)} | "
      f"distintas: {len(distintas)}")
display(distintas if len(distintas) else "Todas las cifras coinciden con la memoria")